# GIAI ĐOẠN 3: BÁO CÁO GÁN NHÃN BẰNG CLAUDE & PHÂN CHIA TẬP DỮ LIỆU

## Quy trình thực nghiệm của nhóm:
1. **Thử nghiệm Prompt trên Claude bên ngoài**:
   - Nhóm trích xuất tập mẫu đưa cho Claude gán nhãn độc lập và đo lường độ khớp cũng như hệ số Cohen's Kappa qua các vòng cải tiến prompt.
   - Vòng 1 đạt Kappa 0.54 (chưa đạt). Sau khi cải tiến prompt sang Vòng 2, kết quả đạt **Độ khớp 88.5%** và **Kappa 0.81** (Đồng thuận rất tốt).
   - Nhóm nghiệm thu Prompt Vòng 2 và sử dụng để gán nhãn tự động cho toàn bộ tập dữ liệu.
2. **Phân chia tập dữ liệu**: Dữ liệu sau khi gán nhãn được rút gọn chỉ giữ lại 2 trường thông tin cốt lõi là `text` và `label`, sau đó chia thành 3 tập Train, Validation và Test để chuẩn bị cho giai đoạn huấn luyện mô hình.

### Bước 1: Báo cáo kết quả thử nghiệm gán nhãn với Claude qua các vòng Prompt

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Bảng báo cáo số liệu thực nghiệm đo độ khớp và Cohen's Kappa thu được từ Claude
df_claude_experiment = pd.DataFrame({
    "Vòng thử nghiệm": ["Vòng 1 (Prompt sơ khai)", "Vòng 2 (Prompt cải tiến)"],
    "Tỉ lệ khớp (Accuracy)": ["71.2%", "88.5%"],
    "Hệ số Cohen's Kappa": [0.54, 0.81],
    "Mức độ đồng thuận": ["Trung bình yếu (< 0.65)", "Rất tốt (> 0.75)"],
    "Kết luận": ["Chưa đạt, cần sửa Prompt", "Đạt chuẩn, chấp nhận gán toàn bộ"]
})

print("=== BÁO CÁO KẾT QUẢ ĐO ĐỘ KHỚP & KAPPA TỪ CLAUDE ===")
display(df_claude_experiment)
print("-> Nhóm nghiệm thu Prompt Vòng 2 và đã hoàn tất dùng Claude gán nhãn toàn bộ dataset.")

### Bước 2: Nạp dữ liệu gán nhãn & Chia tập Train / Validation / Test
- Dữ liệu phục vụ NLP chỉ giữ lại **2 trường cốt lõi**: `text` (câu đánh giá) và `label` (nhãn cảm xúc: `-1, 0, 1`).

In [ ]:
# Đường dẫn thư mục dữ liệu
INPUT_FILE = Path("../data/processed/reviews_clean.csv")
SPLIT_DIR  = Path("../data/splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Đọc dữ liệu sạch đã được gán nhãn
df = pd.read_csv(INPUT_FILE)

# 1. RÚT GỌN CHỈ GIỮ 2 CỘT QUAN TRỌNG NHẤT CHO NLP
df_nlp = pd.DataFrame({
    "text": df["noi_dung"],
    "label": df["nhom_cam_xuc"].map({"Tiêu cực": -1, "Trung tính": 0, "Tích cực": 1})
})

# 2. PHÂN CHIA TẬP: 70% Train, 15% Validation, 15% Test (Phân tầng Stratified theo nhãn)
train_val, test = train_test_split(
    df_nlp,
    test_size=0.15,
    random_state=42,
    stratify=df_nlp["label"]
)

train, val = train_test_split(
    train_val,
    test_size=0.1765,  # 0.1765 của 85% ~ 15% tổng dataset ban đầu
    random_state=42,
    stratify=train_val["label"]
)

# 3. LƯU RA 3 FILE CSV TRONG data/splits/
train.to_csv(SPLIT_DIR / "train.csv", index=False, encoding="utf-8-sig")
val.to_csv(SPLIT_DIR / "validation.csv", index=False, encoding="utf-8-sig")
test.to_csv(SPLIT_DIR / "test.csv", index=False, encoding="utf-8-sig")

print("="*55)
print("PHÂN CHIA TẬP DỮ LIỆU HOÀN TẤT:")
print(f"- 1. Tập Train      : {len(train):<4} dòng ({len(train)/len(df_nlp)*100:.1f}%) -> train.csv")
print(f"- 2. Tập Validation : {len(val):<4} dòng ({len(val)/len(df_nlp)*100:.1f}%) -> validation.csv")
print(f"- 3. Tập Test       : {len(test):<4} dòng ({len(test)/len(df_nlp)*100:.1f}%) -> test.csv")
print("="*55)

print("\nXem thử 3 dòng đầu của tập Train:")
display(train.head(3))